In [ ]:
import os
import sys
import json
import re
import random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

# ── 1. KAGGLE ENVIRONMENT PATCH ──
import __main__
__main__.__file__ = "kaggle_notebook.py"
with open("kaggle_notebook.py", "w") as f:
    f.write("# Physical file for transformers inspection")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── 2. CONFIGURATION ──
MODEL_NAME    = "t5-small"
MAX_INPUT_LEN = 256
MAX_TARGET_LEN= 128
BATCH_SIZE    = 8
NUM_EPOCHS    = 10
LEARNING_RATE = 3e-4

DATA_DIR    = Path("/kaggle/input/datasets/chaitanyajx1/datasetnlp")
REST_FILE   = DATA_DIR / "restaurant_train.jsonl"
LAP_FILE    = DATA_DIR / "laptop_train.jsonl"
MODEL_DIR   = Path("/kaggle/working/aste_cnn_model")

# ── 3. CNN-ENHANCED T5 MODEL (ROBUST VERSION) ──

class T5WithCNN(T5ForConditionalGeneration):
    def __init__(self, config):
        super().__init__(config)
        # CNN acts as a local classifier for word relationships
        self.cnn = nn.Conv1d(
            in_channels=config.d_model, 
            out_channels=config.d_model, 
            kernel_size=3, 
            padding=1
        )
        self.gate = nn.GELU()
        self.layer_norm = nn.LayerNorm(config.d_model)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        head_mask=None,
        decoder_head_mask=None,
        cross_attn_head_mask=None,
        encoder_outputs=None,
        past_key_values=None,
        inputs_embeds=None,
        decoder_inputs_embeds=None,
        labels=None,
        use_cache=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        # 1. Standard Encoder Pass
        if encoder_outputs is None:
            encoder_outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                inputs_embeds=inputs_embeds,
                head_mask=head_mask,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
            )
        
        # 2. Extract Hidden States
        hidden = encoder_outputs[0] if isinstance(encoder_outputs, tuple) else encoder_outputs.last_hidden_state
        
        # 3. CNN Classification Layer (with Residual Connection)
        cnn_in = hidden.transpose(1, 2)  # (B, Dim, Seq)
        cnn_out = self.cnn(cnn_in)
        cnn_out = self.gate(cnn_out)
        cnn_out = cnn_out.transpose(1, 2) # (B, Seq, Dim)
        
        # Add original hidden states back (Residual) and Normalize
        enhanced_hidden = self.layer_norm(hidden + cnn_out)
        
        # 4. Update encoder_outputs
        if isinstance(encoder_outputs, tuple):
            encoder_outputs = (enhanced_hidden,) + encoder_outputs[1:]
        else:
            encoder_outputs.last_hidden_state = enhanced_hidden
            
        # 5. Pass EVERYTHING to the decoder
        return super().forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            head_mask=head_mask,
            decoder_head_mask=decoder_head_mask,
            cross_attn_head_mask=cross_attn_head_mask,
            encoder_outputs=encoder_outputs,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            decoder_inputs_embeds=decoder_inputs_embeds,
            labels=labels,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

# ── 4. DATA PROCESSING ──
def parse_triplets(text):
    triplets = []
    # Regex designed to be more forgiving with spaces
    pattern = r"\(\s*(.+?)\s*\|\s*(.+?)\s*\|\s*([\d.]+#[\d.]+)\s*\)"
    for m in re.finditer(pattern, text):
        triplets.append({
            "Aspect": m.group(1).strip().lower(),
            "Opinion": m.group(2).strip().lower(),
            "VA": m.group(3).strip()
        })
    return triplets

def load_data():
    def to_seq2seq(rec):
        text = re.sub(r'\s+', ' ', rec["Text"]).strip()
        # Ensure we use "Quadruplet" matching your JSON
        quads = rec.get("Quadruplet", [])
        tgt = " ; ".join([f"( {q.get('Aspect','NULL')} | {q.get('Opinion','NULL')} | {q.get('VA','5.00#5.00')} )" for q in quads])
        return f"extract triplets: {text}", tgt

    raw = []
    for path in [REST_FILE, LAP_FILE]:
        if os.path.exists(path):
            with open(path, "r") as f:
                for line in f: raw.append(json.loads(line))
    
    pairs = [to_seq2seq(r) for r in raw]
    random.shuffle(pairs)
    idx = int(len(pairs) * 0.9)
    return pairs[:idx], pairs[idx:]

# ── 5. PREPARATION ──
train_pairs, val_pairs = load_data()
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=True)

class ASTEDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        inp, tgt = self.data[i]
        in_enc = tokenizer(inp, max_length=MAX_INPUT_LEN, padding="max_length", truncation=True, return_tensors="pt")
        tg_enc = tokenizer(tgt, max_length=MAX_TARGET_LEN, padding="max_length", truncation=True, return_tensors="pt")
        labels = tg_enc["input_ids"].squeeze()
        labels[labels == tokenizer.pad_token_id] = -100
        return {"input_ids": in_enc["input_ids"].squeeze(), "attention_mask": in_enc["attention_mask"].squeeze(), "labels": labels}

model = T5WithCNN.from_pretrained(MODEL_NAME).to(DEVICE)

# ── 6. TRAINING ──
args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    # --- STABILITY FIXES ---
    fp16=False,               # T5 is more stable in FP32
    learning_rate=1e-4,       # Slightly lower for the new CNN
    max_grad_norm=1.0,        # Prevents exploding gradients (The "Clip")
    # -----------------------
    load_best_model_at_end=True,
    logging_steps=50,         # Log more often so you catch 'nan' early
    report_to="none"
)
trainer = Seq2SeqTrainer(
    model=model, args=args, 
    train_dataset=ASTEDataset(train_pairs), 
    eval_dataset=ASTEDataset(val_pairs),
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
)

print(f"🚀 Training with {len(train_pairs)} samples...")
trainer.train()

# ── 7. FINAL METRICS ──
def run_evaluation(model, val_data):
    model.eval()
    correct, total_gold, total_pred = 0, 0, 0
    
    print("📊 Computing Final Performance Metrics...")
    for inp_text, gold_text in tqdm(val_data):
        inputs = tokenizer(inp_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=5)
        
        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        preds = parse_triplets(pred_text)
        golds = parse_triplets(gold_text)
        
        gold_set = {(g["Aspect"], g["Opinion"]) for g in golds}
        pred_set = {(p["Aspect"], p["Opinion"]) for p in preds}
        
        total_gold += len(gold_set)
        total_pred += len(pred_set)
        correct += len(gold_set.intersection(pred_set))

    p = correct / total_pred if total_pred > 0 else 0
    r = correct / total_gold if total_gold > 0 else 0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0
    
    print("\n" + "═"*40)
    print(f"  TRIPLET PRECISION: {p:.4f}")
    print(f"  TRIPLET RECALL:    {r:.4f}")
    print(f"  TRIPLET F1-SCORE:  {f1:.4f}")
    print("═"*40)

run_evaluation(model, val_pairs)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5WithCNN LOAD REPORT from: t5-small
Key               | Status  | 
------------------+---------+-
cnn.bias          | MISSING | 
layer_norm.weight | MISSING | 
cnn.weight        | MISSING | 
layer_norm.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Training with 5724 samples...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


KeyboardInterrupt: 

In [4]:
import os
import sys
import json
import re
import random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

# 1. ── KAGGLE PATCH & ENVIRONMENT ──
import __main__
__main__.__file__ = "kaggle_notebook.py"
with open("kaggle_notebook.py", "w") as f:
    f.write("# Stability Patch for Transformers")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── 2. CONFIGURATION (REDUCED FOR STABILITY) ──
MODEL_NAME    = "t5-small"
MAX_INPUT_LEN = 256
MAX_TARGET_LEN= 128
BATCH_SIZE    = 8
NUM_EPOCHS    = 10
LEARNING_RATE = 1e-4  # Lowered from 3e-4 to prevent nan
LOGGING_STEPS = 50

DATA_DIR    = Path("/kaggle/input/datasets/chaitanyajx1/datasetnlp")
REST_FILE   = DATA_DIR / "restaurant_train.jsonl"
LAP_FILE    = DATA_DIR / "laptop_train.jsonl"
MODEL_DIR   = Path("/kaggle/working/aste_cnn_model")

# ── 3. CNN-ENHANCED T5 MODEL ──
class T5WithCNN(T5ForConditionalGeneration):
    def __init__(self, config):
        super().__init__(config)
        self.cnn = nn.Conv1d(config.d_model, config.d_model, kernel_size=3, padding=1)
        self.gate = nn.GELU()
        self.layer_norm = nn.LayerNorm(config.d_model)

    def forward(self, input_ids=None, attention_mask=None, decoder_input_ids=None, 
                decoder_attention_mask=None, head_mask=None, decoder_head_mask=None, 
                cross_attn_head_mask=None, encoder_outputs=None, past_key_values=None, 
                inputs_embeds=None, decoder_inputs_embeds=None, labels=None, 
                use_cache=None, output_attentions=None, output_hidden_states=None, 
                return_dict=None):

        if encoder_outputs is None:
            encoder_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **{"return_dict": True})
        
        hidden = encoder_outputs.last_hidden_state
        
        # 1D-CNN Classification Path
        cnn_out = self.cnn(hidden.transpose(1, 2)).transpose(1, 2)
        cnn_out = self.gate(cnn_out)
        
        # Residual + Norm for stability
        enhanced_hidden = self.layer_norm(hidden + cnn_out)
        encoder_outputs.last_hidden_state = enhanced_hidden
            
        return super().forward(
            input_ids=input_ids, attention_mask=attention_mask, decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask, labels=labels, encoder_outputs=encoder_outputs,
            use_cache=use_cache, return_dict=return_dict
        )

# ── 4. DATASET UTILS ──
def load_data():
    def to_seq2seq(rec):
        text = re.sub(r'\s+', ' ', rec.get("Text", "")).strip()
        quads = rec.get("Quadruplet", [])
        tgt = " ; ".join([f"( {q.get('Aspect','NULL')} | {q.get('Opinion','NULL')} | {q.get('VA','5.00#5.00')} )" for q in quads])
        return f"extract triplets: {text}", tgt

    raw = []
    for path in [REST_FILE, LAP_FILE]:
        if os.path.exists(path):
            with open(path, "r") as f:
                for line in f: raw.append(json.loads(line))
    
    pairs = [to_seq2seq(r) for r in raw]
    random.shuffle(pairs)
    idx = int(len(pairs) * 0.9)
    return pairs[:idx], pairs[idx:]

# ── 5. PREPARATION ──
train_pairs, val_pairs = load_data()
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=True)

class ASTEDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        inp, tgt = self.data[i]
        in_enc = tokenizer(inp, max_length=MAX_INPUT_LEN, padding="max_length", truncation=True, return_tensors="pt")
        tg_enc = tokenizer(tgt, max_length=MAX_TARGET_LEN, padding="max_length", truncation=True, return_tensors="pt")
        labels = tg_enc["input_ids"].squeeze()
        labels[labels == tokenizer.pad_token_id] = -100
        return {"input_ids": in_enc["input_ids"].squeeze(), "attention_mask": in_enc["attention_mask"].squeeze(), "labels": labels}

# Initializing Model
model = T5WithCNN.from_pretrained(MODEL_NAME).to(DEVICE)

# Weight Init for CNN (Crucial for nan prevention)
nn.init.xavier_uniform_(model.cnn.weight)
if model.cnn.bias is not None:
    nn.init.constant_(model.cnn.bias, 0)

# ── 6. TRAINING (FP32 VERSION) ──
args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    # --- STABILITY SETTINGS ---
    fp16=False,               # Disable FP16 to stop nan loss
    learning_rate=LEARNING_RATE,
    max_grad_norm=1.0,        # Gradient clipping
    weight_decay=0.01,
    # --------------------------
    load_best_model_at_end=True,
    logging_steps=LOGGING_STEPS,
    report_to="none",
    predict_with_generate=True
)

trainer = Seq2SeqTrainer(
    model=model, args=args, 
    train_dataset=ASTEDataset(train_pairs), 
    eval_dataset=ASTEDataset(val_pairs),
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
)

print(f"🚀 Training Stably in FP32 Mode...")
trainer.train()

# ── 7. FINAL METRICS ──
def parse_triplets(text):
    triplets = []
    pattern = r"\(\s*(.+?)\s*\|\s*(.+?)\s*\|\s*([\d.]+#[\d.]+)\s*\)"
    for m in re.finditer(pattern, text):
        triplets.append({"Aspect": m.group(1).strip().lower(), "Opinion": m.group(2).strip().lower()})
    return triplets

def run_evaluation(model, val_data):
    model.eval()
    correct, total_gold, total_pred = 0, 0, 0
    print("📊 Evaluating...")
    for inp_text, gold_text in tqdm(val_data):
        inputs = tokenizer(inp_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=3)
        
        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        preds = parse_triplets(pred_text)
        golds = parse_triplets(gold_text)
        
        gold_set = {(g["Aspect"], g["Opinion"]) for g in golds}
        pred_set = {(p["Aspect"], p["Opinion"]) for p in preds}
        
        total_gold += len(gold_set)
        total_pred += len(pred_set)
        correct += len(gold_set.intersection(pred_set))

    p = correct / total_pred if total_pred > 0 else 0
    r = correct / total_gold if total_gold > 0 else 0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0
    print(f"\nFINAL RESULTS: P={p:.4f}, R={r:.4f}, F1={f1:.4f}")

run_evaluation(model, val_pairs)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5WithCNN LOAD REPORT from: t5-small
Key               | Status  | 
------------------+---------+-
cnn.bias          | MISSING | 
layer_norm.weight | MISSING | 
cnn.weight        | MISSING | 
layer_norm.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Training Stably in FP32 Mode...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


KeyboardInterrupt: 